# Ideas 3+4 midterm baseline: tree search + Qwen2.5-VL reader

Runs the question-dependent half of the combined idea on the 24 dev videos (72 questions):

| Condition | Tree | Search scores |
|---|---|---|
| `tree_mm` | multimodal (video + subtitles) | SigLIP visual + MiniLM subtitle |
| `tree_mm_vis` | multimodal | SigLIP visual only |
| `tree_vo` | video only | SigLIP visual only |
| `uniform_check` | none | lmms-eval's 32 uniform frames (should reproduce Karan's `uniform`) |

`blind` and `uniform` come from Karan's results file (same reader, same protocol), so we don't rerun them.
Every condition gives the reader exactly 32 frames, and the reader sees frames only (no subtitles), as in lmms-eval's Video-MME protocol.
So any difference between conditions comes from **which** frames were chosen.

Search settings (3 chapters → 4 sections → 8 frames each) are fixed in advance. Don't tune them on these results.

**Before you start:** Runtime → Change runtime type → **A100** (or L4). Upload Karan's `qwen25_reader.zip` to the top of your Google Drive (`MyDrive/qwen25_reader.zip`).
Then run the cells top to bottom. Everything slow is cached in Drive, so if Colab disconnects, just rerun from the top.

## 1. Install and set up

In [ ]:
!pip -q install transformers==4.57.1 qwen-vl-utils==0.0.14 accelerate decord pyarrow tyro sentencepiece protobuf
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, csv, time, shutil, subprocess, zipfile
DRIVE = '/content/drive/MyDrive/mmml_tree_baseline'     # cache + final results (survives disconnects)
REPO_DIR = '/content/vua'
READER_DIR = '/content/qwen25_reader'
VIDEO_DIR = '/content/videos'
BUNDLE = '/content/bundle'                              # frames + manifest for the reader (local disk is faster)
for d in [DRIVE, f'{DRIVE}/cache', VIDEO_DIR, BUNDLE]:
    os.makedirs(d, exist_ok=True)

if not os.path.exists(REPO_DIR):
    !git clone -q -b VL-temporal-boundries https://github.com/arun-reddy-a/Video-Understanding-Agents.git {REPO_DIR}
if not os.path.exists(READER_DIR):
    !unzip -q -o /content/drive/MyDrive/qwen25_reader.zip -d /content/
print(os.listdir(READER_DIR))

## 2. Search code (writes `tree_search.py`)

In [ ]:
%%writefile /content/tree_search.py
"""Coarse-to-fine search over the temporal trees (combined Ideas 3+4, midterm baseline).

For one question and one tree, pick a fixed budget of frames (default 32):
  1. score every chapter, keep the top K_CHAPTERS
  2. score every section inside those chapters, keep the top K_SECTIONS
  3. inside each kept section, keep the FRAMES_PER_SECTION best-matching 1-fps frames
     (at least MIN_GAP_S seconds apart, so they are not near-duplicates)
  4. if the kept sections cannot supply the whole budget, fill from the next-best sections
  5. return the chosen seconds in time order (the reader sees no timestamps)

Scoring a node (chapter or section):
  visual score   = mean of the node's TOP_FRAMES highest SigLIP similarities between
                   the question text and its 1-fps frames
  subtitle score = cosine similarity between the question (+ options) and the node's
                   subtitle summary, both embedded with MiniLM
  Each score is z-scored across the nodes being compared, because raw scores from
  different encoders are not on the same scale.
  use_subtitles=False -> z(visual)            (video-only search)
  use_subtitles=True  -> z(visual) + z(subtitle) (multimodal search)

Settings are fixed in advance and are not tuned on Video-MME.
"""
from __future__ import annotations

import numpy as np

K_CHAPTERS = 3
K_SECTIONS = 4
FRAMES_PER_SECTION = 8
BUDGET = K_SECTIONS * FRAMES_PER_SECTION  # 32
MIN_GAP_S = 2
TOP_FRAMES = 8


def zscore(values):
    values = np.asarray(values, dtype=float)
    spread = values.std()
    if len(values) < 2 or spread < 1e-8:
        return np.zeros_like(values)
    return (values - values.mean()) / spread


def node_seconds(node, num_seconds):
    """1-fps frame indices that fall inside a node's [start, end) span"""
    start = max(0, int(np.floor(node["start"])))
    end = min(num_seconds, int(np.ceil(node["end"])))
    return np.arange(start, max(start + 1, end)) if start < num_seconds else np.array([], dtype=int)


def visual_score(node, frame_sims):
    seconds = node_seconds(node, len(frame_sims))
    if len(seconds) == 0:
        return -1.0
    sims = np.sort(frame_sims[seconds])[::-1]
    return float(sims[:TOP_FRAMES].mean())


def rank_nodes(nodes, frame_sims, subtitle_sims, use_subtitles):
    """Return (nodes sorted best-first, combined scores in the same order)"""
    visual = zscore([visual_score(node, frame_sims) for node in nodes])
    combined = visual.copy()
    if use_subtitles:
        has_text = np.array([bool(node.get("subtitle_text", "").strip()) for node in nodes])
        raw = np.array([subtitle_sims.get(node["id"], 0.0) for node in nodes])
        text = np.zeros(len(nodes))
        if has_text.sum() >= 2:
            text[has_text] = zscore(raw[has_text])
        combined = visual + text
    order = np.argsort(-combined, kind="stable")
    return [nodes[i] for i in order], combined[order]


def pick_frames_in_section(section, frame_sims, k, taken):
    """Greedy: best-matching seconds first, skipping any within MIN_GAP_S of one already taken"""
    seconds = node_seconds(section, len(frame_sims))
    chosen = []
    for second in seconds[np.argsort(-frame_sims[seconds], kind="stable")]:
        if len(chosen) == k:
            break
        if all(abs(int(second) - other) >= MIN_GAP_S for other in taken + chosen):
            chosen.append(int(second))
    return chosen


def search(tree, frame_sims, subtitle_sims=None, use_subtitles=False, budget=BUDGET):
    """tree: one tree JSON (dict). frame_sims: SigLIP similarity of the question to each
    1-fps frame (index = second). subtitle_sims: {node id: MiniLM similarity}.
    Returns the chosen seconds plus a trace for the failure analysis."""
    frame_sims = np.asarray(frame_sims, dtype=float)
    subtitle_sims = subtitle_sims or {}
    by_id = {node["id"]: node for node in tree["nodes"]}
    chapters = [node for node in tree["nodes"] if node["level"] == 1]

    ranked_chapters, chapter_scores = rank_nodes(chapters, frame_sims, subtitle_sims, use_subtitles)
    kept_chapters = ranked_chapters[:K_CHAPTERS]

    candidate_sections = [by_id[child] for chapter in kept_chapters for child in chapter["children"]]
    ranked_sections, section_scores = rank_nodes(candidate_sections, frame_sims, subtitle_sims, use_subtitles)

    # Sections from the chapters that were not kept, as a fallback to fill the budget
    other_sections = [node for node in tree["nodes"]
                      if node["level"] == 2 and node["id"] not in {s["id"] for s in candidate_sections}]
    if other_sections:
        ranked_other, _ = rank_nodes(other_sections, frame_sims, subtitle_sims, use_subtitles)
    else:
        ranked_other = []

    taken, used_sections = [], []
    for section in ranked_sections[:K_SECTIONS]:
        picked = pick_frames_in_section(section, frame_sims, FRAMES_PER_SECTION, taken)
        taken += picked
        used_sections.append(section["id"])

    for section in ranked_sections[K_SECTIONS:] + ranked_other:
        if len(taken) >= budget:
            break
        picked = pick_frames_in_section(section, frame_sims, budget - len(taken), taken)
        if picked:
            taken += picked
            used_sections.append(section["id"])

    seconds = sorted(taken[:budget])
    covered = sum(by_id[s]["end"] - by_id[s]["start"] for s in used_sections)
    return {
        "seconds": seconds,
        "chapters": [chapter["id"] for chapter in kept_chapters],
        "sections": used_sections,
        "section_spans": [(by_id[s]["start"], by_id[s]["end"]) for s in used_sections],
        "covered_seconds": float(covered),
        "top_chapter_score_gap": float(chapter_scores[0] - chapter_scores[1]) if len(chapter_scores) > 1 else 0.0,
    }

In [ ]:
sys.path.insert(0, '/content')
sys.path.insert(0, READER_DIR)
from tree_search import search
from lmms_protocol.qwen_video_frames import qwen_max_pixels, resize_like_qwen, lmms_uniform_indices

## 3. Questions for the 24 dev videos

In [ ]:
import pandas as pd, numpy as np
from huggingface_hub import hf_hub_download

dev = list(csv.DictReader(open(f'{REPO_DIR}/analysis/VL-temporal-boundary/manifests/long_stratified_24.csv')))
tree_id = {row['videoID']: row['video_id'] for row in dev}       # YouTube id -> tree file name
videos = sorted(tree_id)

try:
    meta_path = hf_hub_download('lmms-lab/Video-MME', 'videomme/test-00000-of-00001.parquet', repo_type='dataset')
except Exception:
    meta_path = hf_hub_download('lmms-eval/Video-MME', 'videomme/test-00000-of-00001.parquet', repo_type='dataset', revision='ead1408')
meta = pd.read_parquet(meta_path)
qs = meta[meta.videoID.isin(videos)].reset_index(drop=True)
print(len(qs), 'questions on', qs.videoID.nunique(), 'videos')   # expect 72 on 24

karan = [json.loads(line) for line in open(f'{READER_DIR}/example/qwen25_long_results.jsonl')]
karan_qids = {r['qid'] for r in karan if r['video'] in tree_id}
assert karan_qids == set(qs.question_id), 'question ids do not match Karan\'s results'
print('question ids match Karan\'s results file')

## 4. Download the 24 videos (~7.4 GB, pulled one file at a time from the HF zips)

In [ ]:
from huggingface_hub import HfFileSystem
index = json.load(open(f'{REPO_DIR}/ArunReddy/analysis_assignment_2/meta/zip_index.json'))
where = {k.split('/')[-1].rsplit('.', 1)[0]: (k, v[0]) for k, v in index.items()}

fs, zips = HfFileSystem(), {}
for vid in videos:
    dst = f'{VIDEO_DIR}/{vid}.mp4'
    if os.path.exists(dst):
        continue
    name, chunk = where[vid]
    if chunk not in zips:
        zips[chunk] = zipfile.ZipFile(fs.open(f'datasets/lmms-lab/Video-MME/videos_chunked_{chunk:02d}.zip', 'rb', block_size=8 << 20))
    t0 = time.time()
    with zips[chunk].open(name) as src, open(dst + '.part', 'wb') as out:
        shutil.copyfileobj(src, out, length=8 << 20)
    os.rename(dst + '.part', dst)
    print(f'{vid}  {os.path.getsize(dst)/1e6:.0f} MB  {time.time()-t0:.0f}s')
print(len(os.listdir(VIDEO_DIR)), 'videos ready')

## 5. Index each video once: SigLIP embeddings of 1-fps frames

This is the query-agnostic part (together with Sanay's tree). Times are saved for the cost table.

In [ ]:
import torch
from concurrent.futures import ThreadPoolExecutor
from transformers import AutoModel, AutoProcessor, AutoTokenizer

SIGLIP = 'google/siglip-so400m-patch14-384'
sig_model = AutoModel.from_pretrained(SIGLIP, dtype=torch.float16).cuda().eval()
sig_proc = AutoProcessor.from_pretrained(SIGLIP)
SIZE = 384
MEAN = torch.tensor(sig_proc.image_processor.image_mean, device='cuda').view(1, 3, 1, 1)
STD = torch.tensor(sig_proc.image_processor.image_std, device='cuda').view(1, 3, 1, 1)

def decode_1fps(vid):
    t0 = time.time()
    cmd = ['ffmpeg', '-v', 'error', '-threads', '0', '-i', f'{VIDEO_DIR}/{vid}.mp4',
           '-vf', f'fps=1,scale={SIZE}:{SIZE}', '-f', 'rawvideo', '-pix_fmt', 'rgb24', 'pipe:']
    raw = subprocess.run(cmd, capture_output=True, check=True).stdout
    return vid, np.frombuffer(raw, np.uint8).reshape(-1, SIZE, SIZE, 3), time.time() - t0

@torch.no_grad()
def embed_frames(frames, batch=128):
    out = []
    for i in range(0, len(frames), batch):
        x = torch.from_numpy(frames[i:i+batch]).cuda().permute(0, 3, 1, 2).float() / 255
        x = ((x - MEAN) / STD).half()
        out.append(torch.nn.functional.normalize(sig_model.get_image_features(pixel_values=x).float(), dim=-1).cpu())
    return torch.cat(out).numpy().astype(np.float16)

todo = [v for v in videos if not os.path.exists(f'{DRIVE}/cache/{v}.npz')]
with ThreadPoolExecutor(max_workers=3) as pool:          # decode 3 videos on CPU while the GPU embeds
    for vid, frames, decode_s in pool.map(decode_1fps, todo):
        t0 = time.time()
        emb = embed_frames(frames)
        np.savez(f'{DRIVE}/cache/{vid}.npz', emb=emb, decode_s=decode_s, embed_s=time.time() - t0)
        print(f'{vid}: {len(emb)} frames  decode {decode_s:.0f}s  embed {time.time()-t0:.0f}s')
print('all videos indexed')

## 6. Question and node text embeddings

In [ ]:
MINILM = 'sentence-transformers/all-MiniLM-L6-v2'          # same text encoder Sanay used for the tree
mini_tok = AutoTokenizer.from_pretrained(MINILM)
mini = AutoModel.from_pretrained(MINILM).cuda().eval()

@torch.no_grad()
def minilm(texts, batch=64):
    out = []
    for i in range(0, len(texts), batch):
        b = mini_tok(texts[i:i+batch], padding=True, truncation=True, max_length=256, return_tensors='pt').to('cuda')
        h = mini(**b).last_hidden_state
        m = b['attention_mask'].unsqueeze(-1).float()
        out.append(torch.nn.functional.normalize((h * m).sum(1) / m.sum(1), dim=-1).cpu().numpy())
    return np.concatenate(out)

@torch.no_grad()
def siglip_text(texts):
    b = sig_proc.tokenizer(texts, padding='max_length', max_length=64, truncation=True, return_tensors='pt').to('cuda')
    return torch.nn.functional.normalize(sig_model.get_text_features(**b).float(), dim=-1).cpu().numpy()

TREES = f'{REPO_DIR}/baseline/VL-temporal-boundary/eda_24_results/trees'
trees = {mode: {vid: json.load(open(f'{TREES}/{mode}/{tree_id[vid]}.json')) for vid in videos}
         for mode in ['multimodal', 'video_only']}

# Node summaries are part of the index (query-agnostic): embed them once per video
node_text_s, node_emb = {}, {}
for vid in videos:
    t0 = time.time()
    nodes = [n for n in trees['multimodal'][vid]['nodes'] if n.get('subtitle_text', '').strip()]
    embs = minilm([n['subtitle_text'] for n in nodes])
    node_emb[vid] = {n['id']: e for n, e in zip(nodes, embs)}
    node_text_s[vid] = time.time() - t0

qs['options_list'] = qs.options.apply(list)
q_sig = siglip_text(list(qs.question))                                     # visual search: question text
q_mini = minilm([q + ' ' + ' '.join(o) for q, o in zip(qs.question, qs.options_list)])  # subtitle search: question + options
print(q_sig.shape, q_mini.shape)

## 7. Run the search for every question and condition

In [ ]:
from decord import VideoReader, cpu

CONDITIONS = {'tree_mm': ('multimodal', True), 'tree_mm_vis': ('multimodal', False), 'tree_vo': ('video_only', False)}
emb = {vid: np.load(f'{DRIVE}/cache/{vid}.npz')['emb'].astype(np.float32) for vid in videos}
video_info = {}
for vid in videos:
    vr = VideoReader(f'{VIDEO_DIR}/{vid}.mp4', ctx=cpu(0))
    video_info[vid] = (len(vr), vr.get_avg_fps())

traces, manifest = [], []
for i, q in qs.iterrows():
    vid = q.videoID
    frame_sims = emb[vid] @ q_sig[i]
    num_frames, fps = video_info[vid]
    conditions = {'uniform_check': [f'n{k}' for k in lmms_uniform_indices(num_frames, fps)]}
    for name, (mode, use_subs) in CONDITIONS.items():
        sub_sims = {nid: float(e @ q_mini[i]) for nid, e in node_emb[vid].items()} if use_subs else {}
        t0 = time.time()
        result = search(trees[mode][vid], frame_sims, sub_sims, use_subtitles=use_subs)
        result.update(qid=q.question_id, video=vid, condition=name, search_s=time.time() - t0)
        traces.append(result)
        conditions[name] = [f'g{s}' for s in result['seconds']]
    manifest.append({'qid': q.question_id, 'video': vid, 'task': q.task_type, 'question': q.question,
                     'options': q.options_list, 'answer': q.answer,
                     'native_fps': fps, 'num_frames': num_frames, 'conditions': conditions})

assert all(len(m['conditions'][c]) == 32 for m in manifest for c in CONDITIONS)
json.dump(traces, open(f'{DRIVE}/search_traces.json', 'w'), indent=1)
print(len(manifest), 'questions,', len(traces), 'searches')

## 8. Export the chosen frames exactly the way the reader expects

In [ ]:
from PIL import Image
for vid in videos:
    num_frames, fps = video_info[vid]
    keys = sorted({k for m in manifest if m['video'] == vid for ks in m['conditions'].values() for k in ks})
    out_dir = f'{BUNDLE}/frames/{vid}'
    os.makedirs(out_dir, exist_ok=True)
    keys = [k for k in keys if not os.path.exists(f'{out_dir}/{k}.jpg')]
    if not keys:
        continue
    native = [int(k[1:]) if k[0] == 'n' else min(round(int(k[1:]) * fps), num_frames - 1) for k in keys]
    vr = VideoReader(f'{VIDEO_DIR}/{vid}.mp4', ctx=cpu(0))
    max_pixels = qwen_max_pixels(num_frames, fps)
    for start in range(0, len(keys), 32):
        frames = vr.get_batch(native[start:start+32]).asnumpy()
        for key, frame in zip(keys[start:start+32], resize_like_qwen(frames, max_pixels)):
            Image.fromarray(frame).save(f'{out_dir}/{key}.jpg', quality=95)
    print(vid, len(keys), 'frames')

with open(f'{BUNDLE}/manifest.jsonl', 'w') as f:
    for m in manifest:
        f.write(json.dumps(m) + '\n')
shutil.copy(f'{BUNDLE}/manifest.jsonl', f'{DRIVE}/manifest.jsonl')

## 9. Run the reader (288 reads; free GPU memory first)

In [ ]:
del sig_model, mini
import gc; gc.collect(); torch.cuda.empty_cache()

# Resume: copy any earlier results back from Drive so finished reads are skipped
if os.path.exists(f'{DRIVE}/reader_results.jsonl') and not os.path.exists(f'{BUNDLE}/reader_results.jsonl'):
    shutil.copy(f'{DRIVE}/reader_results.jsonl', f'{BUNDLE}/reader_results.jsonl')

In [ ]:
%cd {READER_DIR}
!python read_videomme.py --bundle-dir {BUNDLE} --conditions uniform_check tree_mm tree_mm_vis tree_vo
shutil.copy(f'{BUNDLE}/reader_results.jsonl', f'{DRIVE}/reader_results.jsonl')

## 10. Accuracy table (with Karan's `blind` and `uniform` on the same 72 questions)

In [ ]:
%cd {READER_DIR}
ours = [json.loads(line) for line in open(f'{BUNDLE}/reader_results.jsonl')]
dev_q = set(qs.question_id)
karan_dev = [r for r in karan if r['qid'] in dev_q and r['condition'] in ('blind', 'uniform')]
with open(f'{DRIVE}/combined_results.jsonl', 'w') as f:
    for r in karan_dev + ours:
        f.write(json.dumps(r) + '\n')

# Sanity check: our uniform frames should give (almost) the same answers as Karan's uniform
k_uni = {r['qid']: r['pred'] for r in karan_dev if r['condition'] == 'uniform'}
o_uni = {r['qid']: r['pred'] for r in ours if r['condition'] == 'uniform_check'}
print(f"uniform_check agrees with Karan's uniform on {np.mean([k_uni[q] == o_uni[q] for q in o_uni]):.0%} of questions\n")

!python summarize_videomme.py --results-path {DRIVE}/combined_results.jsonl --baseline uniform

## 11. Cost, per-task accuracy, and failure-analysis candidates

In [ ]:
res = pd.DataFrame(karan_dev + ours)
tr = pd.DataFrame(traces)

# Accuracy by task type
by_task = res.pivot_table(index='task', columns='condition', values='correct', aggfunc='mean').round(2)
by_task['n'] = res[res.condition == 'uniform'].groupby('task').size()
display(by_task)

# Confidence in the correct answer and tokens
display(res.groupby('condition')[['correct', 'p_correct', 'n_tokens', 'seconds']].mean().round(3))

# Coverage: how much of the video the chosen sections span
display(tr.groupby('condition')['covered_seconds'].describe().round(0))

# Cost: index once per video, then search + read per question
idx = pd.DataFrame([{**{k: float(v) for k, v in np.load(f'{DRIVE}/cache/{v_}.npz').items() if k != 'emb'},
                     'video': v_, 'node_text_s': node_text_s[v_]} for v_ in videos])
idx['index_s'] = idx.decode_s + idx.embed_s + idx.node_text_s
print(f"index per video: {idx.index_s.mean():.1f}s (decode {idx.decode_s.mean():.1f} + SigLIP {idx.embed_s.mean():.1f} + node text {idx.node_text_s.mean():.2f}); add Sanay's tree-build time")
print(f"search per question: {tr.search_s.mean()*1000:.2f} ms")
print(f"index cost per question with 3 questions/video: {idx.index_s.mean()/3:.1f}s")

by_task.to_csv(f'{DRIVE}/accuracy_by_task.csv'); idx.to_csv(f'{DRIVE}/index_cost.csv'); res.to_csv(f'{DRIVE}/all_results.csv', index=False)

In [ ]:
# Paired outcomes: multimodal tree vs uniform, and multimodal tree vs video-only tree
wide = res.pivot_table(index='qid', columns='condition', values='correct', aggfunc='first')
for a, b in [('tree_mm', 'uniform'), ('tree_mm', 'tree_vo'), ('tree_mm', 'tree_mm_vis')]:
    print(f'{a} vs {b}:'); print(pd.crosstab(wide[a], wide[b], rownames=[a], colnames=[b])); print()

# Questions to inspect by hand: uniform right but tree_mm wrong (losses) and the reverse (wins)
q_by_id = qs.set_index('question_id')
tmm = tr[tr.condition == 'tree_mm'].set_index('qid')
for label, mask in [('LOSS (uniform right, tree_mm wrong)', (wide.uniform == 1) & (wide.tree_mm == 0)),
                    ('WIN (tree_mm right, uniform wrong)', (wide.uniform == 0) & (wide.tree_mm == 1))]:
    print('=' * 20, label, int(mask.sum()))
    for qid in wide[mask].index:
        q, t = q_by_id.loc[qid], tmm.loc[qid]
        spans = ', '.join(f'{a/60:.1f}-{b/60:.1f} min' for a, b in t.section_spans[:4])
        print(f'{qid} [{q.task_type}] {q.question}\n   answer {q.answer}: {[o for o in q.options if o.startswith(q.answer)][0]}\n   chose sections at {spans}\n')